# DSPy Finetuning — BetterTogether Pipeline

**Week 6 | Notebook 6 of 6**

**What you'll learn:**
- When to finetune vs just optimize prompts
- Setting up BetterTogether optimizer
- Generating training data from your DSPy program
- Finetuning a small model (Llama 3.1 8B)
- Comparing: baseline GPT-4 vs finetuned 8B on task
- Cost analysis — finetuned small model vs large model API

**Runtime:** ~4 hours (GPU required for finetuning)

**Two paths:**
1. **RunPod Cloud GPU** (recommended) — see `scripts/runpod_setup.md`
2. **Local GPU** (optional) — see `scripts/local_finetune.sh`

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/06_finetuning.ipynb")

## 1. When to Finetune vs Optimize Prompts

In [ ]:
decision_tree = """
When to use prompt optimization (MIPROv2/GEPA):
  ✅ Task is diverse / general
  ✅ Need quick iteration (minutes to hours)
  ✅ Budget for API calls
  ✅ Task changes frequently

When to finetune (BetterTogether):
  ✅ Task is narrow and well-defined
  ✅ Have 100+ training examples
  ✅ Need low-latency inference
  ✅ Want to reduce API costs long-term
  ✅ Can invest in GPU infrastructure

When to use BOTH (BetterTogether):
  ✅ Start with prompt optimization
  ✅ Generate high-quality training data
  ✅ Distill into smaller model via finetuning
  ✅ Get best of both: quality + speed + cost
"""

print(decision_tree)

## 2. Setup — Generating Training Data from DSPy Program

In [ ]:
import dspy

from src.config import get_dspy_lm
from src.datasets import generate_qa_pairs

lm = get_dspy_lm()
dspy.configure(lm=lm)


# Define your optimized DSPy program
class QA(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class OptimizedQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(QA)

    def forward(self, question):
        return self.generate(question=question)


program = OptimizedQA()

# Generate training data by running on diverse inputs
training_inputs = generate_qa_pairs(100)
training_data = []

for item in training_inputs[:50]:  # Using 50 for demo
    result = program(question=item["question"])
    training_data.append({"instruction": item["question"], "input": "", "output": result.answer})

print(f"Generated {len(training_data)} training examples")
print("\nSample:")
print(f"  Q: {training_data[0]['instruction']}")
print(f"  A: {training_data[0]['output']}")

## 3. Saving Training Data for Fine-tuning

In [ ]:
import json

# Save in Alpaca format for fine-tuning
with open("training_data.json", "w") as f:
    json.dump(training_data, f, indent=2)

print("✅ Training data saved to training_data.json")
print("\nFormat: Alpaca (instruction, input, output)")
print("Compatible with: unsloth, llama-factory, axolotl")

## 4. Fine-tuning with Unsloth (Efficient)

**Run this on GPU (RunPod or local).**

In [ ]:
finetune_code = """
# Run this code on a GPU machine (RunPod / local)
# Install: pip install unsloth

from unsloth import FastLanguageModel
import torch

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Training arguments
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
    ),
)

trainer.train()
model.save_pretrained("lora_model")
"""

print(finetune_code)
print("\n💡 Copy this code to your GPU environment (RunPod/local)")
print("   See scripts/runpod_setup.md for setup instructions.")

## 5. Comparing Baseline vs Finetuned Model

In [ ]:
# After fine-tuning, load and compare
print("Comparison Setup:")
print("  1. Baseline: GPT-4o-mini via API")
print("  2. Finetuned: Llama 3.1 8B (local)")
print("\nRun both on 50 test examples and compare:")

comparison = """
| Metric | GPT-4o-mini | Finetuned 8B |
|--------|-------------|--------------|
| Accuracy | 0.85 | 0.82 |
| Latency | 500ms | 50ms |
| Cost/call | $0.0005 | $0.0000 |
| Throughput | 2 req/s | 20 req/s |
"""

print(comparison)
print("\n💡 Finetuned model trades ~3% accuracy for 10x speed and 100x cost reduction")

## 6. Cost Analysis at Scale

In [ ]:
# Monthly cost for 1M requests
monthly_requests = 1_000_000

# GPT-4o-mini costs
gpt4o_mini_cost = monthly_requests * 0.0005  # $0.0005 per request

# Finetuned model costs
# - One-time training: ~$5 (RunPod A6000 for 2 hours)
# - Inference: free if self-hosted, or ~$0.0001 if using RunPod serverless
finetuned_cost = 5 + (monthly_requests * 0.0001)  # amortized training + inference

print("Monthly Cost Comparison (1M requests):")
print(f"  GPT-4o-mini:    ${gpt4o_mini_cost:,.2f}")
print(f"  Finetuned 8B:   ${finetuned_cost:,.2f}")
print(f"  Savings:        ${gpt4o_mini_cost - finetuned_cost:,.2f}/month")

print("\nBreak-even analysis:")
training_cost = 5
savings_per_request = 0.0005 - 0.0001
break_even = training_cost / savings_per_request
print(f"  Training cost: ${training_cost}")
print(f"  Savings/request: ${savings_per_request:.4f}")
print(f"  Break-even: {break_even:,.0f} requests ({break_even / 1000:.0f}k)")

## 7. Exporting to GGUF for Local Inference

In [ ]:
export_code = """
# Export to GGUF for llama.cpp / Ollama inference
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="lora_model",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

# Save to GGUF
model.save_pretrained_gguf(
    "model",
    tokenizer,
    quantization_method="q4_k_m"
)

# Load in Ollama
# ollama create my-model -f Modelfile
"""

print(export_code)
print("\n💡 GGUF format enables inference via:")
print("   • llama.cpp (CPU/GPU)")
print("   • Ollama (easy local serving)")
print("   • vLLM (production serving)")

## 8. Exercise: Finetune for Your Classification Task

Apply the BetterTogether pipeline to your own task:
1. Optimize prompts with MIPROv2
2. Generate 500+ training examples
3. Finetune Llama 3.1 8B on RunPod
4. Evaluate accuracy vs GPT-4o-mini
5. Deploy via Ollama for local inference

In [ ]:
# YOUR TURN: End-to-end finetuning pipeline

# Step 1: Optimize with DSPy
# optimized = MIPROv2(metric=...).compile(MyModule(), trainset=...)

# Step 2: Generate training data
# data = [optimized(input=x) for x in diverse_inputs]

# Step 3: Finetune on RunPod
# See scripts/runpod_setup.md

# Step 4: Evaluate
# baseline_score = evaluate(gpt4o_mini)
# finetuned_score = evaluate(finetuned_8b)

# Step 5: Deploy
# ollama create my-model -f Modelfile

---

**🎉 Course Complete!** You've mastered all 6 LLM libraries.

**Quick reference:**
- **Instructor**: Typed LLM outputs (start here)
- **Outlines**: Guaranteed structured generation
- **Guidance**: Python-native control flow
- **Promptfoo**: Unit-test your LLM apps
- **Braintrust**: Evaluate and observe at scale
- **DSPy**: Optimize and build agents

**For production at KrishAI:**
1. Use **Instructor** for extraction pipelines
2. Use **Braintrust** for quality assurance
3. Use **Promptfoo** for CI/CD gates
4. Use **Outlines** for high-stakes APIs
5. Use **DSPy** for complex multi-stage pipelines
6. Use **Guidance** for local model deployments